In [ ]:
WORKSPACE_IDs = ['2b1d454b-61a1-4abb-96c3-2b476d945a96']

StatementMeta(, ba0d0b44-7547-473b-91f3-f6ddc81d3993, 8, Finished, Available, Finished, False)

# 1 Imports

In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
import sempy.dependencies as dependencies
import sempy.fabric as fabric

import pandas as pd


import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model
import sempy.fabric as fabric
import sempy_labs.report as rep
from sempy_labs.report import ReportWrapper




StatementMeta(, ba0d0b44-7547-473b-91f3-f6ddc81d3993, 10, Finished, Available, Finished, False)

# 2 Configuration & Helper Functions

In [ ]:


def sanitize_column_names(df):
    df.columns = df.columns.str.lower().str.replace(" ", "_").str.replace("-", "_")
    return df

def add_composite_key(df, key_columns, pk_name='pk', separator='|'):
    """
    Add human-readable composite primary key from specified columns
    
    Args:
        df: pandas DataFrame
        key_columns: list of column names to concatenate
        pk_name: name for the primary key column (default: 'pk')
        separator: delimiter between key components (default: '|')
    
    Returns:
        DataFrame with primary key as first column
    """
    # Create composite key by concatenating columns
    df[pk_name] = df[key_columns].astype(str).agg(separator.join, axis=1)
    
    # Move PK to first column
    cols = [pk_name] + [c for c in df.columns if c != pk_name]
    return df[cols]


StatementMeta(, ba0d0b44-7547-473b-91f3-f6ddc81d3993, 11, Finished, Available, Finished, False)

## Primary Key Schema

| Table | Primary Key Name | Composite Key Columns | Format |
|-------|-----------------|----------------------|---------|
| t_fabric_artifacts | artifact_pk | id + workspace_id | `{id}\|{workspace_id}` |
| t_dataset_relations | relation_pk | relationship_name + dataset_id | `{relationship_name}\|{dataset_id}` |
| t_dataset_columns | column_pk | table_name + column_name + dataset_id | `{table_name}\|{column_name}\|{dataset_id}` |
| t_dataset_tables | table_pk | name + dataset_id | `{name}\|{dataset_id}` |
| t_dataset_partitions | partition_pk | table_name + partition_name + dataset_id | `{table_name}\|{partition_name}\|{dataset_id}` |
| t_dataset_measures | measure_pk | measure_name + dataset_id | `{measure_name}\|{dataset_id}` |
| t_dataset_dependencies | dependency_pk | object_key + referenced_object_key | `{object_key}\|{referenced_object_key}` |
| t_report_metadata | report_pk | report_id | `{report_id}` |
| t_report_pages | page_pk | report_id + page_name | `{report_id}\|{page_name}` |
| t_report_visuals | visual_pk | report_id + page_name + visual_name | `{report_id}\|{page_name}\|{visual_name}` |
| t_report_semantic_model_objects | report_object_pk | object_key + dataset_id | `{object_key}\|{dataset_id}` |
| t_lakehouse_tables | lakehouse_table_pk | table_name + lakehouse_id | `{table_name}\|{lakehouse_id}` |
| t_lakehouse_columns | lakehouse_column_pk | table_name + column_name + lakehouse_id | `{table_name}\|{column_name}\|{lakehouse_id}` |

### Additional Key Columns
- **t_dataset_dependencies**: Also includes `object_key` and `referenced_object_key` columns that reference the primary keys from tables/columns/measures
- **t_report_pages**: Also includes `page_display_name` (human-readable page name)
- **t_report_semantic_model_objects**: 
  - Includes `object_key` (references dataset object's primary key: table_pk, column_pk, or measure_pk)
  - Includes `dataset_id` (references the semantic model)
  - Includes `report_source` (indicates whether the object is tracked at 'page' or 'visual' level - 'page' for Page Filters, 'visual' for all other sources)
  - Includes `report_object_sk` (surrogate key that references either page_pk from t_report_pages when source is 'Page Filter', or visual_pk from t_report_visuals for all other sources)
- **t_lakehouse_tables**: Includes `lakehouse_id`, `lakehouse_name`, and `workspace_id` for reference
- **t_lakehouse_columns**: Includes `lakehouse_id`, `lakehouse_name`, `table_name`, and `workspace_id` for reference

# 3. Get all artifacts from selected Workspaces

In [5]:
df_fabric_artifacts = pd.DataFrame()
for ws in WORKSPACE_IDs:
        if(df_fabric_artifacts.empty):
            df_fabric_artifacts=fabric.list_items(workspace=ws)
        else:
            df_fabric_artifacts=pd.concat([df_fabric_artifacts, fabric.list_items(workspace=ws)], axis=0, ignore_index=True)

df_fabric_artifacts = sanitize_column_names(df_fabric_artifacts)
df_fabric_artifacts = add_composite_key(df_fabric_artifacts, ['id', 'workspace_id'], 'artifact_pk')
df_fabric_artifacts
spark.createDataFrame(df_fabric_artifacts).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("t_fabric_artifacts")

StatementMeta(, ba0d0b44-7547-473b-91f3-f6ddc81d3993, 13, Finished, Available, Finished, False)

In [ ]:
display(df_fabric_artifacts)

StatementMeta(, ba0d0b44-7547-473b-91f3-f6ddc81d3993, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cc265e2b-21aa-4ac8-8825-309b00be8147)

# 4. Get all infos for semantic models

In [6]:
df_semantic_models = df_fabric_artifacts[df_fabric_artifacts['type']=="SemanticModel"]
df_semantic_models
spark.createDataFrame(df_semantic_models).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("t_dataset_semantic_models")

StatementMeta(, 3a9bed7b-6d8a-4277-8b92-738f2d127ea8, 11, Finished, Available, Finished, False)

StatementMeta(, ba0d0b44-7547-473b-91f3-f6ddc81d3993, 15, Finished, Available, Finished, False)

In [15]:
df_semantic_models

StatementMeta(, ba0d0b44-7547-473b-91f3-f6ddc81d3993, 40, Finished, Available, Finished, False)

,artifact_pk,id,display_name,description,type,workspace_id,folder_id
4,80490e11-fa1f-4d7c-a5e3-fefa2d877221|2b1d454b-...,80490e11-fa1f-4d7c-a5e3-fefa2d877221,Training Datamodel Solution,,SemanticModel,2b1d454b-61a1-4abb-96c3-2b476d945a96,None
5,90ce1fe5-5ea0-41d1-acc0-53d307f5f846|2b1d454b-...,90ce1fe5-5ea0-41d1-acc0-53d307f5f846,Almost empty,,SemanticModel,2b1d454b-61a1-4abb-96c3-2b476d945a96,None
6,623a4171-5730-4246-b21d-c5538b642a26|2b1d454b-...,623a4171-5730-4246-b21d-c5538b642a26,Human Resources Sample PBIX,,SemanticModel,2b1d454b-61a1-4abb-96c3-2b476d945a96,None
7,e8ff5f16-3695-4b13-8fa2-1ecf8fea38ac|2b1d454b-...,e8ff5f16-3695-4b13-8fa2-1ecf8fea38ac,HealthcareDirectLake,,SemanticModel,2b1d454b-61a1-4abb-96c3-2b476d945a96,None
8,88096254-b4ed-470a-abc2-87af6c9dca3c|2b1d454b-...,88096254-b4ed-470a-abc2-87af6c9dca3c,Sample DirectQuery,,SemanticModel,2b1d454b-61a1-4abb-96c3-2b476d945a96,None
9,4380425f-4750-4abb-9c74-5126febfe2bb|2b1d454b-...,4380425f-4750-4abb-9c74-5126febfe2bb,Test_Query,,SemanticModel,2b1d454b-61a1-4abb-96c3-2b476d945a96,None


## 4.1 Get CalcDependencies

In [ ]:
df_relations = pd.DataFrame()

for index, row in df_semantic_models.iterrows():

    df_new = pd.DataFrame()
    df_new = fabric.list_relationships(dataset=row['id'], workspace= row['workspace_id'])
    df_new['dataset_id'] = row['id']
    df_new['workspace_id'] = row['workspace_id']
    if (df_relations.empty):
        df_relations = df_new
    else:
        df_relations=pd.concat([df_relations, df_new], axis=0, ignore_index=True)

df_relations = sanitize_column_names(df_relations)
df_relations = add_composite_key(df_relations, ['relationship_name', 'dataset_id'], 'relation_pk')

spark.createDataFrame(df_relations).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("t_dataset_relations")

df_relations

StatementMeta(, 3a9bed7b-6d8a-4277-8b92-738f2d127ea8, 12, Finished, Available, Finished, False)

,relation_pk,multiplicity,from_table,from_column,to_table,to_column,active,cross_filtering_behavior,security_filtering_behavior,join_on_date_behavior,rely_on_referential_integrity,state,modified_time,relationship_name,dataset_id,workspace_id
0,337a2a65-a524-5200-ffa1-fe63f49de233|80490e11-...,m:1,Sales,ProductKey,Product,ProductKey,True,OneDirection,OneDirection,DateAndTime,False,Ready,2024-06-21 20:15:34,337a2a65-a524-5200-ffa1-fe63f49de233,80490e11-fa1f-4d7c-a5e3-fefa2d877221,2b1d454b-61a1-4abb-96c3-2b476d945a96
1,54e760c0-ed91-374c-3ce7-e372fa7fa483|80490e11-...,m:1,Sales,SalesTerritoryKey,Region,SalesTerritoryKey,True,OneDirection,OneDirection,DateAndTime,False,Ready,2024-06-21 20:15:34,54e760c0-ed91-374c-3ce7-e372fa7fa483,80490e11-fa1f-4d7c-a5e3-fefa2d877221,2b1d454b-61a1-4abb-96c3-2b476d945a96
2,0cc9b6a8-4c34-8503-dd93-c0291078795d|80490e11-...,m:1,Sales,ResellerKey,Reseller,ResellerKey,True,OneDirection,OneDirection,DateAndTime,False,Ready,2024-06-21 20:15:34,0cc9b6a8-4c34-8503-dd93-c0291078795d,80490e11-fa1f-4d7c-a5e3-fefa2d877221,2b1d454b-61a1-4abb-96c3-2b476d945a96
3,b67f057a-6409-6474-fce4-595e8035683f|80490e11-...,m:1,Targets,EmployeeID,Salesperson,EmployeeID,True,OneDirection,OneDirection,DateAndTime,False,Ready,2024-06-21 20:15:34,b67f057a-6409-6474-fce4-595e8035683f,80490e11-fa1f-4d7c-a5e3-fefa2d877221,2b1d454b-61a1-4abb-96c3-2b476d945a96
4,26b1374a-2310-f5fa-2053-7602cf86a4b9|80490e11-...,m:1,Sales,EmployeeKey,Salesperson,EmployeeKey,True,OneDirection,OneDirection,DateAndTime,False,Ready,2025-07-04 13:27:34,26b1374a-2310-f5fa-2053-7602cf86a4b9,80490e11-fa1f-4d7c-a5e3-fefa2d877221,2b1d454b-61a1-4abb-96c3-2b476d945a96
5,cc2e0e7c-5b82-65cc-30a8-f822bbac58eb|80490e11-...,m:1,Sales,OrderDate,Date,Date,True,OneDirection,OneDirection,DateAndTime,False,Ready,2025-07-04 13:55:06,cc2e0e7c-5b82-65cc-30a8-f822bbac58eb,80490e11-fa1f-4d7c-a5e3-fefa2d877221,2b1d454b-61a1-4abb-96c3-2b476d945a96
6,4b28c4f5-2d51-4740-a3a6-32733b5b9da4|80490e11-...,m:1,Targets,TargetMonth,Date,Date,True,OneDirection,OneDirection,DateAndTime,False,Ready,2025-07-04 13:55:06,4b28c4f5-2d51-4740-a3a6-32733b5b9da4,80490e11-fa1f-4d7c-a5e3-fefa2d877221,2b1d454b-61a1-4abb-96c3-2b476d945a96
7,7ddd41c5-a1f5-477b-9712-84fdc7313c7d|623a4171-...,m:1,Employee,date,Date,Date,True,OneDirection,OneDirection,DateAndTime,False,Ready,2017-03-27 21:48:24,7ddd41c5-a1f5-477b-9712-84fdc7313c7d,623a4171-5730-4246-b21d-c5538b642a26,2b1d454b-61a1-4abb-96c3-2b476d945a96
8,497871a2-f5e1-40ff-90d7-6f980ca1119a|623a4171-...,m:1,Employee,FP,FP,FP,True,OneDirection,OneDirection,DateAndTime,False,Ready,2017-03-27 21:48:24,497871a2-f5e1-40ff-90d7-6f980ca1119a,623a4171-5730-4246-b21d-c5538b642a26,2b1d454b-61a1-4abb-96c3-2b476d945a96
9,20c07ef2-2c50-4fd4-ad56-8b0e0e36ddc4|623a4171-...,m:1,Employee,EthnicGroup,Ethnicity,Ethnic Group,True,OneDirection,OneDirection,DateAndTime,False,Ready,2017-03-27 21:48:24,20c07ef2-2c50-4fd4-ad56-8b0e0e36ddc4,623a4171-5730-4246-b21d-c5538b642a26,2b1d454b-61a1-4abb-96c3-2b476d945a96


In [ ]:
df_datasources = pd.DataFrame()

for index, row in df_semantic_models.iterrows():

    df_new = pd.DataFrame()
    df_new = labs.list_semantic_model_datasources(dataset=row['id'], workspace= row['workspace_id'])
    df_new['dataset_id'] = row['id']
    df_new['workspace_id'] = row['workspace_id']
    if (df_datasources.empty):    
        df_datasources = df_new
    else:
        df_datasources=pd.concat([df_datasources, df_new], axis=0, ignore_index=True)

df_datasources = sanitize_column_names(df_datasources)
df_datasources
df_datasources = add_composite_key(df_datasources, ['datasource_id', 'connection_server', 'dataset_id'], 'datasource_pk')

spark.createDataFrame(df_datasources).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("t_dataset_datasources")


StatementMeta(, 3a9bed7b-6d8a-4277-8b92-738f2d127ea8, 15, Finished, Available, Finished, False)

In [20]:
df_datasources_directlake = pd.DataFrame()

for index, row in df_semantic_models.iterrows():
    try:
        res =  labs.directlake.get_direct_lake_sources(dataset=row['id'], workspace= row['workspace_id'])
        if res is not None and len(res)>0:
            df_new = pd.DataFrame(res)
            df_new['dataset_id'] = row['id']
            df_new['workspace_id'] = row['workspace_id']
            if (df_datasources_directlake.empty):    
                df_datasources_directlake = df_new
            else:
                df_datasources_directlake=pd.concat([df_datasources_directlake, df_new], axis=0, ignore_index=True)
    except Exception as e:
        print(f"Error processing dataset {row['id']} in workspace {row['workspace_id']}: {e}")


display(df_datasources_directlake)
df_datasources_directlake = sanitize_column_names(df_datasources_directlake)

spark.createDataFrame(df_datasources_directlake).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("t_dataset_directlake_datasources")

StatementMeta(, ba0d0b44-7547-473b-91f3-f6ddc81d3993, 50, Finished, Available, Finished, False)

Error processing dataset 4380425f-4750-4abb-9c74-5126febfe2bb in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96: 400 The request could not be processed due to missing or invalid information for url: https://api.fabric.microsoft.com/metadata/artifacts/LH_Bronze
Headers: {'Cache-Control': 'no-store, must-revalidate, no-cache', 'Pragma': 'no-cache', 'Transfer-Encoding': 'chunked', 'Content-Type': 'application/octet-stream', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains', 'X-Frame-Options': 'deny', 'X-Content-Type-Options': 'nosniff', 'RequestId': 'e835dd8b-d8df-4ae0-b56d-00b753b42e45', 'Access-Control-Expose-Headers': 'RequestId', 'Date': 'Tue, 09 Jun 2026 12:07:55 GMT'}


SynapseWidget(Synapse.DataFrame, 961cb61f-c306-4efa-ac8d-0352c46c35f2)

In [ ]:
df_columns = pd.DataFrame()

for index, row in df_semantic_models.iterrows():
    try:
        df_new = pd.DataFrame()
        df_new = fabric.list_columns(dataset=row['id'], workspace= row['workspace_id'], additional_xmla_properties=["LineageTag"])
        df_new['dataset_id'] = row['id']
        df_new['workspace_id'] = row['workspace_id']
        if (df_columns.empty):
            df_columns = df_new
        else:
            df_columns=pd.concat([df_columns, df_new], axis=0, ignore_index=True)
    except:
        pass

df_columns = sanitize_column_names(df_columns)
df_columns = add_composite_key(df_columns, ['table_name', 'column_name', 'dataset_id'], 'column_pk')
df_columns = df_columns.astype(str)
df_columns = df_columns.replace('nan', None)
spark.createDataFrame(df_columns).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("t_dataset_columns")

df_columns

StatementMeta(, ba0d0b44-7547-473b-91f3-f6ddc81d3993, 16, Finished, Available, Finished, False)

,column_pk,table_name,column_name,description,type,data_type,hidden,format_string,source,data_category,...,is_available_in_mdx,encoding_hint,state,error_message,alternate_of_base_column,alternate_of_base_table,modified_time,lineage_tag,dataset_id,workspace_id
0,Salesperson|EmployeeKey|80490e11-fa1f-4d7c-a5e...,Salesperson,EmployeeKey,,Data,Int64,True,0,EmployeeKey,,...,True,Default,Ready,,None,None,2024-06-17 17:16:57,c44af1bf-53ad-476e-b494-3839e0c3204c,80490e11-fa1f-4d7c-a5e3-fefa2d877221,2b1d454b-61a1-4abb-96c3-2b476d945a96
1,Salesperson|EmployeeID|80490e11-fa1f-4d7c-a5e3...,Salesperson,EmployeeID,,Data,Int64,True,0,EmployeeID,,...,True,Default,Ready,,None,None,2024-06-17 17:39:09,29d6a6cb-8d7c-4fac-bd97-824645bff4c3,80490e11-fa1f-4d7c-a5e3-fefa2d877221,2b1d454b-61a1-4abb-96c3-2b476d945a96
2,Salesperson|Title|80490e11-fa1f-4d7c-a5e3-fefa...,Salesperson,Title,,Data,String,False,,Title,,...,True,Default,Ready,,None,None,2024-04-30 19:07:56,8efa946d-14e2-4138-bc8c-5136c2a775be,80490e11-fa1f-4d7c-a5e3-fefa2d877221,2b1d454b-61a1-4abb-96c3-2b476d945a96
3,Salesperson|UPN|80490e11-fa1f-4d7c-a5e3-fefa2d...,Salesperson,UPN,,Data,String,True,,UPN,,...,True,Default,Ready,,None,None,2024-05-10 15:58:17,b7627f0f-d6d9-4bcf-9d3c-dacff4ec08af,80490e11-fa1f-4d7c-a5e3-fefa2d877221,2b1d454b-61a1-4abb-96c3-2b476d945a96
4,Salesperson|Salesperson|80490e11-fa1f-4d7c-a5e...,Salesperson,Salesperson,,Data,String,False,,Salesperson,,...,True,Default,Ready,,None,None,2024-04-30 20:46:39,665b013f-c666-4d6b-bc2c-d3d668ba96c7,80490e11-fa1f-4d7c-a5e3-fefa2d877221,2b1d454b-61a1-4abb-96c3-2b476d945a96
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,ProductSubCategories|CategoryID|4380425f-4750-...,ProductSubCategories,CategoryID,,Data,Int64,False,0,CategoryID,,...,True,Default,Ready,,None,None,2026-06-02 06:39:06,bc3ef0fb-430f-4f68-b60c-66a98dff0917,4380425f-4750-4abb-9c74-5126febfe2bb,2b1d454b-61a1-4abb-96c3-2b476d945a96
242,ProductSubCategories|SubCategoryName|4380425f-...,ProductSubCategories,SubCategoryName,,Data,String,False,,SubCategoryName,,...,True,Default,Ready,,None,None,2026-06-01 20:51:46,cfd390b7-f24b-4309-ab90-f69d19c5527a,4380425f-4750-4abb-9c74-5126febfe2bb,2b1d454b-61a1-4abb-96c3-2b476d945a96
243,ProductSubCategories|DateInserted|4380425f-475...,ProductSubCategories,DateInserted,,Data,DateTime,False,General Date,DateInserted,,...,True,Default,Ready,,None,None,2026-06-01 20:51:46,ffd8c048-668f-4899-b744-eac78f1c93c8,4380425f-4750-4abb-9c74-5126febfe2bb,2b1d454b-61a1-4abb-96c3-2b476d945a96
244,ProductSubCategories|SourceFilename|4380425f-4...,ProductSubCategories,SourceFilename,,Data,String,False,,SourceFilename,,...,True,Default,Ready,,None,None,2026-06-01 20:51:46,fe476bd4-67c3-49d6-8593-53e45cea70f6,4380425f-4750-4abb-9c74-5126febfe2bb,2b1d454b-61a1-4abb-96c3-2b476d945a96


In [8]:
df_tables = pd.DataFrame()

for index, row in df_semantic_models.iterrows():

    df_new = pd.DataFrame()
    df_new = fabric.list_tables(dataset=row['id'], workspace= row['workspace_id'], additional_xmla_properties=["LineageTag"])
    df_new['dataset_id'] = row['id']
    df_new['workspace_id'] = row['workspace_id']
    if (df_tables.empty):
        df_tables = df_new
    else:
        df_tables=pd.concat([df_tables, df_new], axis=0, ignore_index=True)

df_tables = sanitize_column_names(df_tables)
df_tables = add_composite_key(df_tables, ['name', 'dataset_id'], 'table_pk')

# Convert all columns to strings to avoid Spark type inference issues
df_tables = df_tables.astype(str)
df_tables = df_tables.replace('nan', None)

spark.createDataFrame(df_tables).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("t_dataset_tables")

display(df_tables)

StatementMeta(, 3291c773-d509-49e5-a363-18b55b344d05, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bea0df3d-a644-4ef5-ba60-72984e7e133c)

In [9]:
df_partitions = pd.DataFrame()

for index, row in df_semantic_models.iterrows():

    df_new = pd.DataFrame()
    df_new = fabric.list_partitions(dataset=row['id'], workspace= row['workspace_id'])
    df_new['dataset_id'] = row['id']
    df_new['workspace_id'] = row['workspace_id']
    if (df_partitions.empty):
        df_partitions = df_new
    else:
        df_partitions=pd.concat([df_partitions, df_new], axis=0, ignore_index=True)

df_partitions = sanitize_column_names(df_partitions)
df_partitions = add_composite_key(df_partitions, ['table_name', 'partition_name', 'dataset_id'], 'partition_pk')
df_partitions['table_sk'] = df_partitions['table_name'] + '|' + df_partitions['dataset_id']




df_partitions = df_partitions.astype(str)
df_partitions = df_partitions.replace('nan', None)
spark.createDataFrame(df_partitions).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("t_dataset_partitions")

display(df_partitions.head(100))

StatementMeta(, 6dfa0e71-6c23-4894-91ee-d02250b872db, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e0afdb0b-71cd-4223-b7e0-e4232212c716)

StatementMeta(, 3291c773-d509-49e5-a363-18b55b344d05, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5a81a0c9-e36d-4d63-8944-b08cd6eb36f7)

In [10]:
df_measures = pd.DataFrame()

for index, row in df_semantic_models.iterrows():

    df_new = pd.DataFrame()
    df_new = fabric.list_measures(dataset=row['id'], workspace= row['workspace_id'], additional_xmla_properties=["LineageTag"])
    df_new['dataset_id'] = row['id']
    df_new['workspace_id'] = row['workspace_id']
    if (df_measures.empty):
        df_measures = df_new
    else:
        df_measures=pd.concat([df_measures, df_new], axis=0, ignore_index=True)

df_measures = sanitize_column_names(df_measures)
df_measures = add_composite_key(df_measures, ['measure_name', 'dataset_id'], 'measure_pk')

df_measures = df_measures.astype(str)
df_measures = df_measures.replace('nan', None)
spark.createDataFrame(df_measures).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("t_dataset_measures")

StatementMeta(, 6dfa0e71-6c23-4894-91ee-d02250b872db, 17, Finished, Available, Finished, False)

StatementMeta(, 3291c773-d509-49e5-a363-18b55b344d05, 17, Finished, Available, Finished, False)

In [11]:
df_dependencies = pd.DataFrame()

for index, row in df_semantic_models.iterrows():

    df_new = pd.DataFrame()
    df_new = labs.get_model_calc_dependencies(row['id'], row['workspace_id'])
    df_new['dataset_id'] = row['id']
    df_new['workspace_id'] = row['workspace_id']
    if (df_dependencies.empty):
        df_dependencies = df_new
    else:
        df_dependencies=pd.concat([df_dependencies, df_new], axis=0, ignore_index=True)

df_dependencies = sanitize_column_names(df_dependencies)

# Create primary key lookup tables from tables, columns, and measures
# Tables lookup (for Table and Calc Table types)
df_tables_keys = df_tables[['dataset_id', 'name', 'table_pk']].copy()
df_tables_keys.columns = ['dataset_id', 'table_name', 'table_pk']

# Columns lookup (for Column type)
df_columns_keys = df_columns[['dataset_id', 'table_name', 'column_name', 'column_pk']].copy()
df_columns_keys.columns = ['dataset_id', 'table_name', 'object_name', 'column_pk']

# Measures lookup (for Measure type)
df_measures_keys = df_measures[['dataset_id', 'measure_name', 'measure_pk']].copy()
df_measures_keys.columns = ['dataset_id', 'object_name', 'measure_pk']

# Join object_key (for Measure/Column: use dataset_id + table_name + object_name)
df_dependencies = df_dependencies.merge(
    df_columns_keys,
    left_on=['dataset_id', 'table_name', 'object_name'],
    right_on=['dataset_id', 'table_name', 'object_name'],
    how='left',
    suffixes=('', '_col')
)

df_dependencies = df_dependencies.merge(
    df_measures_keys,
    left_on=['dataset_id', 'object_name'],
    right_on=['dataset_id', 'object_name'],
    how='left',
    suffixes=('', '_meas')
)

# Join object_key (for Table/Calc Table: use dataset_id + table_name)
df_dependencies = df_dependencies.merge(
    df_tables_keys,
    left_on=['dataset_id', 'table_name'],
    right_on=['dataset_id', 'table_name'],
    how='left',
    suffixes=('', '_tbl')
)

# Combine primary key columns into object_key
df_dependencies['object_key'] = df_dependencies['column_pk'].fillna(
    df_dependencies['measure_pk']
).fillna(
    df_dependencies['table_pk']
)

# Drop intermediate columns
df_dependencies = df_dependencies.drop(columns=['column_pk', 'measure_pk', 'table_pk'], errors='ignore')

# Join referenced_object_key (for Measure/Column: use dataset_id + referenced_table + referenced_object)
df_dependencies = df_dependencies.merge(
    df_columns_keys,
    left_on=['dataset_id', 'referenced_table', 'referenced_object'],
    right_on=['dataset_id', 'table_name', 'object_name'],
    how='left',
    suffixes=('', '_ref_col')
)

df_dependencies = df_dependencies.merge(
    df_measures_keys,
    left_on=['dataset_id', 'referenced_object'],
    right_on=['dataset_id', 'object_name'],
    how='left',
    suffixes=('', '_ref_meas')
)

# Join referenced_object_key (for Table/Calc Table: use dataset_id + referenced_table)
df_dependencies = df_dependencies.merge(
    df_tables_keys,
    left_on=['dataset_id', 'referenced_table'],
    right_on=['dataset_id', 'table_name'],
    how='left',
    suffixes=('', '_ref_tbl')
)

# Combine primary key columns into referenced_object_key
df_dependencies['referenced_object_key'] = df_dependencies['column_pk'].fillna(
    df_dependencies['measure_pk']
).fillna(
    df_dependencies['table_pk']
)

# Drop intermediate and duplicate columns
df_dependencies = df_dependencies.drop(columns=[
    'column_pk', 'measure_pk', 'table_pk',
    'table_name_ref_col', 'object_name_ref_col',
    'object_name_ref_meas',
    'table_name_ref_tbl'
], errors='ignore')

# Add primary key for dependencies table
df_dependencies = add_composite_key(
    df_dependencies,
    ['object_key', 'referenced_object_key'],
    'dependency_pk'
)

df_dependencies = df_dependencies.astype(str)

# Replace 'nan' strings with NULL
df_dependencies = df_dependencies.replace('nan', None)

spark.createDataFrame(df_dependencies).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("t_dataset_dependencies")

StatementMeta(, 6dfa0e71-6c23-4894-91ee-d02250b872db, 19, Finished, Available, Finished, False)

StatementMeta(, 3291c773-d509-49e5-a363-18b55b344d05, 18, Finished, Available, Finished, False)

In [12]:
# Display sample of dependencies with object keys
print(f"Total dependencies: {len(df_dependencies)}")
print("\nColumns in final dependencies table:")
print(df_dependencies.columns.tolist())
print("\nSample rows with object keys:")
df_dependencies[['dependency_pk', 'table_name', 'object_name', 'object_key', 'referenced_table', 'referenced_object', 'referenced_object_key']].head(10)

StatementMeta(, bbd5c540-71e1-464c-8a3d-1b8681302f4c, 51, Finished, Available, Finished, False)

Total dependencies: 547

Columns in final dependencies table:
['dependency_pk', 'table_name', 'object_name', 'object_type', 'expression', 'referenced_table', 'referenced_object', 'referenced_object_type', 'full_object_name', 'referenced_full_object_name', 'parent_node', 'dataset_id', 'workspace_id', 'object_key', 'referenced_object_key']

Sample rows with object keys:


,dependency_pk,table_name,object_name,object_key,referenced_table,referenced_object,referenced_object_key
0,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221|P...,Product,Products,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Product,Category,Product|Category|80490e11-fa1f-4d7c-a5e3-fefa2...
1,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221|P...,Product,Products,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Product,Subcategory,Product|Subcategory|80490e11-fa1f-4d7c-a5e3-fe...
2,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221|P...,Product,Products,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Product,Product,Product|Product|80490e11-fa1f-4d7c-a5e3-fefa2d...
3,Region|80490e11-fa1f-4d7c-a5e3-fefa2d877221|Re...,Region,Regions,Region|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Region,Group,Region|Group|80490e11-fa1f-4d7c-a5e3-fefa2d877221
4,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221|...,Reseller,Resellers,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Reseller,Reseller,Reseller|Reseller|80490e11-fa1f-4d7c-a5e3-fefa...
5,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221|...,Reseller,Geography,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Reseller,Country-Region,Reseller|Country-Region|80490e11-fa1f-4d7c-a5e...
6,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221|...,Reseller,Geography,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Reseller,State-Province,Reseller|State-Province|80490e11-fa1f-4d7c-a5e...
7,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221|...,Reseller,Geography,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Reseller,City,Reseller|City|80490e11-fa1f-4d7c-a5e3-fefa2d87...
8,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221|...,Reseller,Geography,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Reseller,Reseller,Reseller|Reseller|80490e11-fa1f-4d7c-a5e3-fefa...
9,Sales|80490e11-fa1f-4d7c-a5e3-fefa2d877221|Pro...,Sales,337a2a65-a524-5200-ffa1-fe63f49de233,Sales|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Product,ProductKey,Product|ProductKey|80490e11-fa1f-4d7c-a5e3-fef...


StatementMeta(, a5061fef-47f5-4aad-b55e-ad8d8c97129c, 20, Finished, Available, Finished, False)

Total dependencies: 547

Columns in final dependencies table:
['dependency_pk', 'table_name', 'object_name', 'object_type', 'expression', 'referenced_table', 'referenced_object', 'referenced_object_type', 'full_object_name', 'referenced_full_object_name', 'parent_node', 'dataset_id', 'workspace_id', 'object_key', 'referenced_object_key']

Sample rows with object keys:


...

Total dependencies: 547

Columns in final dependencies table:
['dependency_pk', 'table_name', 'object_name', 'object_type', 'expression', 'referenced_table', 'referenced_object', 'referenced_object_type', 'full_object_name', 'referenced_full_object_name', 'parent_node', 'dataset_id', 'workspace_id', 'object_key', 'referenced_object_key']

Sample rows with object keys:


,dependency_pk,table_name,object_name,object_key,referenced_table,referenced_object,referenced_object_key
0,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221|P...,Product,Products,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Product,Category,Product|Category|80490e11-fa1f-4d7c-a5e3-fefa2...
1,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221|P...,Product,Products,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Product,Subcategory,Product|Subcategory|80490e11-fa1f-4d7c-a5e3-fe...
2,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221|P...,Product,Products,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Product,Product,Product|Product|80490e11-fa1f-4d7c-a5e3-fefa2d...
3,Region|80490e11-fa1f-4d7c-a5e3-fefa2d877221|Re...,Region,Regions,Region|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Region,Group,Region|Group|80490e11-fa1f-4d7c-a5e3-fefa2d877221
4,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221|...,Reseller,Resellers,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Reseller,Reseller,Reseller|Reseller|80490e11-fa1f-4d7c-a5e3-fefa...
5,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221|...,Reseller,Geography,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Reseller,Country-Region,Reseller|Country-Region|80490e11-fa1f-4d7c-a5e...
6,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221|...,Reseller,Geography,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Reseller,State-Province,Reseller|State-Province|80490e11-fa1f-4d7c-a5e...
7,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221|...,Reseller,Geography,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Reseller,City,Reseller|City|80490e11-fa1f-4d7c-a5e3-fefa2d87...
8,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221|...,Reseller,Geography,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Reseller,Reseller,Reseller|Reseller|80490e11-fa1f-4d7c-a5e3-fefa...
9,Sales|80490e11-fa1f-4d7c-a5e3-fefa2d877221|Pro...,Sales,337a2a65-a524-5200-ffa1-fe63f49de233,Sales|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Product,ProductKey,Product|ProductKey|80490e11-fa1f-4d7c-a5e3-fef...


StatementMeta(, 3291c773-d509-49e5-a363-18b55b344d05, 19, Finished, Available, Finished, False)

Total dependencies: 929

Columns in final dependencies table:
['dependency_pk', 'table_name', 'object_name', 'object_type', 'expression', 'referenced_table', 'referenced_object', 'referenced_object_type', 'full_object_name', 'referenced_full_object_name', 'parent_node', 'dataset_id', 'workspace_id', 'object_key', 'referenced_object_key']

Sample rows with object keys:


,dependency_pk,table_name,object_name,object_key,referenced_table,referenced_object,referenced_object_key
0,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221|P...,Product,Products,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Product,Category,Product|Category|80490e11-fa1f-4d7c-a5e3-fefa2...
1,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221|P...,Product,Products,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Product,Subcategory,Product|Subcategory|80490e11-fa1f-4d7c-a5e3-fe...
2,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221|P...,Product,Products,Product|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Product,Product,Product|Product|80490e11-fa1f-4d7c-a5e3-fefa2d...
3,Region|80490e11-fa1f-4d7c-a5e3-fefa2d877221|Re...,Region,Regions,Region|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Region,Group,Region|Group|80490e11-fa1f-4d7c-a5e3-fefa2d877221
4,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221|...,Reseller,Resellers,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Reseller,Reseller,Reseller|Reseller|80490e11-fa1f-4d7c-a5e3-fefa...
5,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221|...,Reseller,Geography,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Reseller,Country-Region,Reseller|Country-Region|80490e11-fa1f-4d7c-a5e...
6,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221|...,Reseller,Geography,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Reseller,State-Province,Reseller|State-Province|80490e11-fa1f-4d7c-a5e...
7,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221|...,Reseller,Geography,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Reseller,City,Reseller|City|80490e11-fa1f-4d7c-a5e3-fefa2d87...
8,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221|...,Reseller,Geography,Reseller|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Reseller,Reseller,Reseller|Reseller|80490e11-fa1f-4d7c-a5e3-fefa...
9,Sales|80490e11-fa1f-4d7c-a5e3-fefa2d877221|Pro...,Sales,337a2a65-a524-5200-ffa1-fe63f49de233,Sales|80490e11-fa1f-4d7c-a5e3-fefa2d877221,Product,ProductKey,Product|ProductKey|80490e11-fa1f-4d7c-a5e3-fef...


# Get Reports

In [13]:
df_reports = df_fabric_artifacts[df_fabric_artifacts['type']=="Report"]
df_reports

StatementMeta(, 3291c773-d509-49e5-a363-18b55b344d05, 20, Finished, Available, Finished, False)

,artifact_pk,id,display_name,description,type,workspace_id,folder_id
0,a8247d4d-8590-4bd6-8521-0f84bbb73a77|2b1d454b-...,a8247d4d-8590-4bd6-8521-0f84bbb73a77,Training Datamodel Solution,,Report,2b1d454b-61a1-4abb-96c3-2b476d945a96,None
1,88e4eb9f-f94f-4618-9225-20921b8f614c|2b1d454b-...,88e4eb9f-f94f-4618-9225-20921b8f614c,Human Resources Sample PBIX,,Report,2b1d454b-61a1-4abb-96c3-2b476d945a96,None
2,92f7fbc4-c952-42fb-b427-feaf4a399ebf|2b1d454b-...,92f7fbc4-c952-42fb-b427-feaf4a399ebf,Sample DirectQuery,,Report,2b1d454b-61a1-4abb-96c3-2b476d945a96,None
3,0548db5e-51d2-427d-9888-be5c1cbca6a7|2b1d454b-...,0548db5e-51d2-427d-9888-be5c1cbca6a7,Test_Query,,Report,2b1d454b-61a1-4abb-96c3-2b476d945a96,None


In [14]:
# Process report metadata first to get dataset_id for each report
df_report_metadata = pd.DataFrame()

for index, row in df_reports.iterrows():
    try:
        print(f"Processing report {row['id']} in workspace {row['workspace_id']}")
        df_new = pd.DataFrame()
        df_new = rep.get_report(row['id'], workspace=row['workspace_id'])

        df_new['workspace_id'] = row['workspace_id']

        if (df_report_metadata.empty):
            df_report_metadata = df_new
        else:
            df_report_metadata=pd.concat([df_report_metadata, df_new], axis=0, ignore_index=True)
    except Exception as e:
        print(f"Error processing report {row['id']} in workspace {row['workspace_id']}: {e}")

df_report_metadata = sanitize_column_names(df_report_metadata)
df_report_metadata = add_composite_key(df_report_metadata, ['report_id'], 'report_pk')

# Convert all columns to strings to avoid Spark type inference issues
df_report_metadata = df_report_metadata.astype(str)
df_report_metadata = df_report_metadata.replace('nan', None)

spark.createDataFrame(df_report_metadata).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("t_report_metadata")  


StatementMeta(, 3291c773-d509-49e5-a363-18b55b344d05, 21, Finished, Available, Finished, False)

Processing report a8247d4d-8590-4bd6-8521-0f84bbb73a77 in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing report 88e4eb9f-f94f-4618-9225-20921b8f614c in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing report 92f7fbc4-c952-42fb-b427-feaf4a399ebf in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing report 0548db5e-51d2-427d-9888-be5c1cbca6a7 in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96


In [15]:
df_report_visuals = pd.DataFrame()

for index, row in df_reports.iterrows():
    try:
        df_new = pd.DataFrame()
        report = rep.ReportWrapper(report=row['id'], workspace=row['workspace_id'])
        df_new = report.list_visuals()
        df_new['report_id'] = row['id']
        df_new['workspace_id'] = row['workspace_id']
    except:
        print(f"Error processing report {row['id']} in workspace {row['workspace_id']}")
    if (df_report_visuals.empty):
        df_report_visuals = df_new
    else:
        df_report_visuals=pd.concat([df_report_visuals, df_new], axis=0, ignore_index=True)

df_report_visuals = sanitize_column_names(df_report_visuals)
df_report_visuals = add_composite_key(df_report_visuals, ['report_id', 'page_name', 'visual_name'], 'visual_pk')

# Convert all columns to strings to avoid Spark type inference issues

df_report_visuals = df_report_visuals.astype(str)
df_report_visuals = df_report_visuals.replace('nan', None)


spark.createDataFrame(df_report_visuals).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("t_report_visuals")  


StatementMeta(, 6dfa0e71-6c23-4894-91ee-d02250b872db, 23, Finished, Available, Finished, False)

Error processing report a8247d4d-8590-4bd6-8521-0f84bbb73a77 in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96


StatementMeta(, 3291c773-d509-49e5-a363-18b55b344d05, 22, Finished, Available, Finished, False)

Error processing report a8247d4d-8590-4bd6-8521-0f84bbb73a77 in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96


In [16]:
# Extract unique pages from visuals
# Include page_display_name if available
page_columns = ['report_id', 'page_name', 'workspace_id']
if 'page_display_name' in df_report_visuals.columns:
    page_columns.append('page_display_name')

df_report_pages = df_report_visuals[page_columns].drop_duplicates().copy()

# Add composite primary key for pages
df_report_pages = add_composite_key(df_report_pages, ['report_id', 'page_name'], 'page_pk')

# Convert all columns to strings to avoid Spark type inference issues
df_report_pages = df_report_pages.astype(str)
df_report_pages = df_report_pages.replace('nan', None)

spark.createDataFrame(df_report_pages).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("t_report_pages")

print(f"Created t_report_pages with {len(df_report_pages)} pages")

StatementMeta(, c57e9096-302f-4840-859e-ed0f510d31cd, 24, Finished, Available, Finished, False)

Created t_report_pages with 6 pages


StatementMeta(, 6dfa0e71-6c23-4894-91ee-d02250b872db, 24, Finished, Available, Finished, False)

Created t_report_pages with 6 pages


StatementMeta(, 3291c773-d509-49e5-a363-18b55b344d05, 23, Finished, Available, Finished, False)

Created t_report_pages with 7 pages


In [17]:
# Process semantic model objects - only for reports that have visuals/pages
# Get unique reports from visuals and pages
df_reports_with_content = pd.concat([
    df_report_visuals[['report_id', 'workspace_id']],
    df_report_pages[['report_id', 'workspace_id']]
]).drop_duplicates()

df_report_semantic_model_objects = pd.DataFrame()
df_report_visual_objects = pd.DataFrame()

for index, row in df_reports_with_content.iterrows():
    try:
        df_new = pd.DataFrame()
        report = rep.ReportWrapper(report=row['report_id'], workspace=row['workspace_id'])
        
        # Get semantic model objects
        df_new = report.list_semantic_model_objects(extended=True)
        df_new['report_id'] = row['report_id']
        df_new['workspace_id'] = row['workspace_id']
        
        # Get visual objects for page/visual name mapping
        df_visual_obj = report.list_visual_objects()
        df_visual_obj['report_id'] = row['report_id']
        df_visual_obj['workspace_id'] = row['workspace_id']
        
        if (df_report_visual_objects.empty):
            df_report_visual_objects = df_visual_obj
        else:
            df_report_visual_objects = pd.concat([df_report_visual_objects, df_visual_obj], axis=0, ignore_index=True)
    except Exception as e:
        print(f"Error processing report {row['report_id']} in workspace {row['workspace_id']}: {e}")
        
    if (df_report_semantic_model_objects.empty):
        df_report_semantic_model_objects = df_new
    else:
        df_report_semantic_model_objects=pd.concat([df_report_semantic_model_objects, df_new], axis=0, ignore_index=True)

df_report_semantic_model_objects = sanitize_column_names(df_report_semantic_model_objects)
df_report_visual_objects = sanitize_column_names(df_report_visual_objects)

# Debug: Check what we received
print(f"Total semantic model objects retrieved: {len(df_report_semantic_model_objects)}")
print(f"Total visual objects retrieved: {len(df_report_visual_objects)}")
if len(df_report_semantic_model_objects) > 0:
    print(f"Semantic model object columns: {df_report_semantic_model_objects.columns.tolist()}")
    print(f"Object types found: {df_report_semantic_model_objects['object_type'].value_counts().to_dict() if 'object_type' in df_report_semantic_model_objects.columns else 'No object_type column'}")
if len(df_report_visual_objects) > 0:
    print(f"Visual object columns: {df_report_visual_objects.columns.tolist()}")

# Join with report_metadata to get dataset_id for each report
# Create a lookup with just report_id and dataset_id
df_report_lookup = df_report_metadata[['report_id', 'dataset_id']].drop_duplicates()

# Check if dataset_id is already in the data; if not, join with report_metadata
if 'dataset_id' not in df_report_semantic_model_objects.columns:
    df_report_semantic_model_objects = df_report_semantic_model_objects.merge(
        df_report_lookup,
        on='report_id',
        how='left'
    )

# Create object_key by looking up the primary key based on object_type
# Split into separate dataframes by object_type for easier processing
df_report_columns = df_report_semantic_model_objects[df_report_semantic_model_objects['object_type'] == 'Column'].copy() if 'object_type' in df_report_semantic_model_objects.columns else pd.DataFrame()
df_report_measures = df_report_semantic_model_objects[df_report_semantic_model_objects['object_type'] == 'Measure'].copy() if 'object_type' in df_report_semantic_model_objects.columns else pd.DataFrame()
df_report_tables = df_report_semantic_model_objects[df_report_semantic_model_objects['object_type'] == 'Table'].copy() if 'object_type' in df_report_semantic_model_objects.columns else pd.DataFrame()

print(f"Split by object_type - Columns: {len(df_report_columns)}, Measures: {len(df_report_measures)}, Tables: {len(df_report_tables)}")

# For Columns: join with df_columns using dataset_id, table_name, and object_name (which is the column_name)
if not df_report_columns.empty:
    print(f"Processing {len(df_report_columns)} column objects...")
    print(f"Sample column data: {df_report_columns[['dataset_id', 'table_name', 'object_name']].head(2).to_dict() if len(df_report_columns) > 0 else 'empty'}")
    df_report_columns = df_report_columns.merge(
        df_columns[['dataset_id', 'table_name', 'column_name', 'column_pk']],
        left_on=['dataset_id', 'table_name', 'object_name'],
        right_on=['dataset_id', 'table_name', 'column_name'],
        how='left',
        suffixes=('', '_lookup')
    )
    df_report_columns['object_key'] = df_report_columns['column_pk']
    print(f"After join - Columns with object_key: {df_report_columns['object_key'].notna().sum()}")
    df_report_columns = df_report_columns.drop(columns=['column_pk', 'column_name'], errors='ignore')

# For Measures: join with df_measures using dataset_id and object_name (which is the measure_name)
if not df_report_measures.empty:
    print(f"Processing {len(df_report_measures)} measure objects...")
    print(f"Sample measure data: {df_report_measures[['dataset_id', 'object_name']].head(2).to_dict() if len(df_report_measures) > 0 else 'empty'}")
    df_report_measures = df_report_measures.merge(
        df_measures[['dataset_id', 'measure_name', 'measure_pk']],
        left_on=['dataset_id', 'object_name'],
        right_on=['dataset_id', 'measure_name'],
        how='left',
        suffixes=('', '_lookup')
    )
    df_report_measures['object_key'] = df_report_measures['measure_pk']
    print(f"After join - Measures with object_key: {df_report_measures['object_key'].notna().sum()}")
    df_report_measures = df_report_measures.drop(columns=['measure_pk', 'measure_name'], errors='ignore')

# For Tables: join with df_tables using dataset_id and object_name (which is the table name)
if not df_report_tables.empty:
    print(f"Processing {len(df_report_tables)} table objects...")
    print(f"Sample table data: {df_report_tables[['dataset_id', 'object_name']].head(2).to_dict() if len(df_report_tables) > 0 else 'empty'}")
    df_report_tables = df_report_tables.merge(
        df_tables[['dataset_id', 'name', 'table_pk']],
        left_on=['dataset_id', 'object_name'],
        right_on=['dataset_id', 'name'],
        how='left',
        suffixes=('', '_lookup')
    )
    df_report_tables['object_key'] = df_report_tables['table_pk']
    print(f"After join - Tables with object_key: {df_report_tables['object_key'].notna().sum()}")
    df_report_tables = df_report_tables.drop(columns=['table_pk', 'name'], errors='ignore')

# Combine all back together
df_report_semantic_model_objects = pd.concat([df_report_columns, df_report_measures, df_report_tables], axis=0, ignore_index=True)
print(f"After combining all object types: {len(df_report_semantic_model_objects)} total rows")

# Check report_source distribution before processing
if 'report_source' in df_report_semantic_model_objects.columns:
    print(f"Report source distribution: {df_report_semantic_model_objects['report_source'].value_counts().to_dict()}")

# Split into Page Filters (page-level), Visual Filters, and other visual-level objects
df_page_filters = df_report_semantic_model_objects[df_report_semantic_model_objects['report_source'] == 'Page Filter'].copy() if 'report_source' in df_report_semantic_model_objects.columns else pd.DataFrame()
df_visual_filters = df_report_semantic_model_objects[df_report_semantic_model_objects['report_source'] == 'Visual Filter'].copy() if 'report_source' in df_report_semantic_model_objects.columns else pd.DataFrame()
df_other_visuals = df_report_semantic_model_objects[~df_report_semantic_model_objects['report_source'].isin(['Page Filter', 'Visual Filter'])].copy() if 'report_source' in df_report_semantic_model_objects.columns else pd.DataFrame()

print(f"Split by report_source - Page Filters: {len(df_page_filters)}, Visual Filters: {len(df_visual_filters)}, Other visuals: {len(df_other_visuals)}")

# Process Page Filters: Use report_source_object as page_name
if len(df_page_filters) > 0:
    print("Processing page-level filters...")
    if 'report_source_object' in df_page_filters.columns:
        # Use report_source_object as the page_name for joining
        df_page_filters['page_name'] = df_page_filters['report_source_object']
        df_page_filters = df_page_filters.merge(
            df_report_pages[['report_id', 'page_name', 'page_pk']],
            on=['report_id', 'page_name'],
            how='left'
        )
        print(f"After page join - Page Filters with page_pk: {df_page_filters['page_pk'].notna().sum()}")
    else:
        print("Warning: Page Filters missing report_source_object column")
        df_page_filters['page_pk'] = None
    
    df_page_filters['report_object_sk'] = df_page_filters['page_pk']
    df_page_filters['visual_pk'] = None

# Process Visual Filters: Use report_source_object for page_name and visual_name
if len(df_visual_filters) > 0:
    print("Processing visual-level filters...")
    if 'report_source_object' in df_visual_filters.columns:
        # For Visual Filters, report_source_object contains the visual identifier
        # We need to extract page_name and visual_name from it or use it directly
        # First, try to join with visual_objects to get the mapping
        if len(df_report_visual_objects) > 0:
            visual_filter_lookup = df_report_visual_objects[['report_id', 'page_name', 'visual_name', 'visual_name']].drop_duplicates()
            # Use report_source_object to match with visual_name from visual_objects
            df_visual_filters = df_visual_filters.merge(
                df_report_visual_objects[['report_id', 'page_name', 'visual_name']].drop_duplicates(),
                left_on=['report_id', 'report_source_object'],
                right_on=['report_id', 'visual_name'],
                how='left',
                suffixes=('_old', '')
            )
            print(f"After visual_objects mapping - Visual Filters with page_name: {df_visual_filters['page_name'].notna().sum()}")
        
        # Join with visuals to get visual_pk
        if 'page_name' in df_visual_filters.columns and 'visual_name' in df_visual_filters.columns:
            df_visual_filters = df_visual_filters.merge(
                df_report_visuals[['report_id', 'page_name', 'visual_name', 'visual_pk']],
                on=['report_id', 'page_name', 'visual_name'],
                how='left'
            )
            print(f"After visual join - Visual Filters with visual_pk: {df_visual_filters['visual_pk'].notna().sum()}")
        else:
            df_visual_filters['visual_pk'] = None
    else:
        print("Warning: Visual Filters missing report_source_object column")
        df_visual_filters['visual_pk'] = None
    
    df_visual_filters['report_object_sk'] = df_visual_filters['visual_pk']

# Process other visual-level objects: Use visual_objects to map page_name and visual_name
if len(df_other_visuals) > 0 and len(df_report_visual_objects) > 0 and 'object_name' in df_report_visual_objects.columns:
    print("Processing other visual-level objects with visual_objects mapping...")
    
    # Create a lookup from visual_objects with object info and page/visual names
    visual_obj_lookup = df_report_visual_objects[['report_id', 'page_name', 'visual_name', 'object_name', 'object_type']].copy()
    
    # Join with visual_objects to get the correct page/visual names
    df_other_visuals = df_other_visuals.merge(
        visual_obj_lookup,
        left_on=['report_id', 'object_name', 'object_type'],
        right_on=['report_id', 'object_name', 'object_type'],
        how='left',
        suffixes=('_old', '')
    )
    
    # Drop old page_name and visual_name if they existed
    df_other_visuals = df_other_visuals.drop(columns=['page_name_old', 'visual_name_old'], errors='ignore')
    
    print(f"After visual_objects mapping - records with visual_name: {df_other_visuals['visual_name'].notna().sum()}")
    
    # Join with visuals to get visual_pk
    if 'page_name' in df_other_visuals.columns and 'visual_name' in df_other_visuals.columns:
        df_other_visuals = df_other_visuals.merge(
            df_report_visuals[['report_id', 'page_name', 'visual_name', 'visual_pk']],
            on=['report_id', 'page_name', 'visual_name'],
            how='left'
        )
        print(f"After visual join - other visuals with visual_pk: {df_other_visuals['visual_pk'].notna().sum()}")
    
    df_other_visuals['report_object_sk'] = df_other_visuals['visual_pk']

# Combine all back together
df_report_semantic_model_objects = pd.concat([df_page_filters, df_visual_filters, df_other_visuals], axis=0, ignore_index=True)
print(f"After recombining: {len(df_report_semantic_model_objects)} total rows")
print(f"Final report_object_sk - populated: {df_report_semantic_model_objects['report_object_sk'].notna().sum()}, null: {df_report_semantic_model_objects['report_object_sk'].isna().sum()}")

# Add composite primary key using object_key + dataset_id
df_report_semantic_model_objects = add_composite_key(df_report_semantic_model_objects, ['object_key', 'dataset_id'], 'report_object_pk')

# Convert all columns to strings to avoid Spark type inference issues
df_report_semantic_model_objects = df_report_semantic_model_objects.astype(str)
df_report_semantic_model_objects = df_report_semantic_model_objects.replace('nan', None)


spark.createDataFrame(df_report_semantic_model_objects).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("t_report_semantic_model_objects")

print(f"Created t_report_semantic_model_objects with {len(df_report_semantic_model_objects)} objects")


StatementMeta(, 3291c773-d509-49e5-a363-18b55b344d05, 24, Finished, Available, Finished, False)

Total semantic model objects retrieved: 122
Total visual objects retrieved: 80
Semantic model object columns: ['table_name', 'object_name', 'object_type', 'report_source', 'report_source_object', 'valid_semantic_model_object', 'report_id', 'workspace_id']
Object types found: {'Column': 78, 'Measure': 44}
Visual object columns: ['page_name', 'page_display_name', 'visual_name', 'table_name', 'object_name', 'object_type', 'implicit_measure', 'sparkline', 'visual_calc', 'format', 'object_display_name', 'report_id', 'workspace_id']
Split by object_type - Columns: 78, Measures: 44, Tables: 0
Processing 78 column objects...
Sample column data: {'dataset_id': {0: '623a4171-5730-4246-b21d-c5538b642a26', 1: '623a4171-5730-4246-b21d-c5538b642a26'}, 'table_name': {0: 'Date', 1: 'Date'}, 'object_name': {0: 'Year', 1: 'Month'}}
After join - Columns with object_key: 78
Processing 44 measure objects...
Sample measure data: {'dataset_id': {33: '623a4171-5730-4246-b21d-c5538b642a26', 39: '623a4171-5730-

In [18]:
df_new = report.list_semantic_model_objects(extended=True)
df_new

StatementMeta(, 3291c773-d509-49e5-a363-18b55b344d05, 25, Finished, Available, Finished, False)

,Table Name,Object Name,Object Type,Report Source,Report Source Object,Valid Semantic Model Object
0,Products,ProductName,Column,Visual Filter,'Page 1'[a4a12234625e46bc8ea0],True
1,Orders,Items sold,Measure,Visual Filter,'Page 1'[a4a12234625e46bc8ea0],True
2,Orders,Sum Sales,Measure,Visual Filter,'Page 1'[a4a12234625e46bc8ea0],True
3,Orders,Sum Discount,Measure,Visual Filter,'Page 1'[a4a12234625e46bc8ea0],True
4,Orders,Units sold with MakeFlag,Measure,Visual Filter,'Page 1'[a4a12234625e46bc8ea0],True
5,ProductCategories,CategoryName,Column,Visual Filter,'Page 1'[d336e7f1928370380bc2],True
6,Orders,Items sold,Measure,Visual Filter,'Page 1'[d336e7f1928370380bc2],True
7,Orders,Sum Sales,Measure,Visual Filter,'Page 1'[d336e7f1928370380bc2],True
8,Orders,Sum Discount,Measure,Visual Filter,'Page 1'[d336e7f1928370380bc2],True
9,Orders,Units sold with MakeFlag,Measure,Visual Filter,'Page 1'[d336e7f1928370380bc2],True


In [19]:
df_newnew = report.list_visual_objects()
df_newnew


StatementMeta(, 3291c773-d509-49e5-a363-18b55b344d05, 26, Finished, Available, Finished, False)

,Page Name,Page Display Name,Visual Name,Table Name,Object Name,Object Type,Implicit Measure,Sparkline,Visual Calc,Format,Object Display Name
0,678afcaeb10d16806236,Page 1,a4a12234625e46bc8ea0,Products,ProductName,Column,False,False,False,None,None
1,678afcaeb10d16806236,Page 1,a4a12234625e46bc8ea0,Orders,Items sold,Measure,False,False,False,None,None
2,678afcaeb10d16806236,Page 1,a4a12234625e46bc8ea0,Orders,Sum Sales,Measure,False,False,False,None,None
3,678afcaeb10d16806236,Page 1,a4a12234625e46bc8ea0,Orders,Sum Discount,Measure,False,False,False,None,None
4,678afcaeb10d16806236,Page 1,a4a12234625e46bc8ea0,Orders,Units sold with MakeFlag,Measure,False,False,False,None,None
5,678afcaeb10d16806236,Page 1,d336e7f1928370380bc2,ProductCategories,CategoryName,Column,False,False,False,None,None
6,678afcaeb10d16806236,Page 1,d336e7f1928370380bc2,Orders,Items sold,Measure,False,False,False,None,None
7,678afcaeb10d16806236,Page 1,d336e7f1928370380bc2,Orders,Sum Sales,Measure,False,False,False,None,None
8,678afcaeb10d16806236,Page 1,d336e7f1928370380bc2,Orders,Sum Discount,Measure,False,False,False,None,None
9,678afcaeb10d16806236,Page 1,d336e7f1928370380bc2,Orders,Units sold with MakeFlag,Measure,False,False,False,None,None


# Get Lakehouses

In [20]:
df_lakehouses = df_fabric_artifacts[df_fabric_artifacts['type']=="Lakehouse"]
df_lakehouses

StatementMeta(, 3291c773-d509-49e5-a363-18b55b344d05, 27, Finished, Available, Finished, False)

,artifact_pk,id,display_name,description,type,workspace_id,folder_id
22,b357c6a6-b3dd-46bd-beb7-32bbe8f9b637|2b1d454b-...,b357c6a6-b3dd-46bd-beb7-32bbe8f9b637,LH_I_need_data,,Lakehouse,2b1d454b-61a1-4abb-96c3-2b476d945a96,None
26,d555b438-935c-4a50-a14b-93d637712a30|2b1d454b-...,d555b438-935c-4a50-a14b-93d637712a30,healthcare2_msft_admin,,Lakehouse,2b1d454b-61a1-4abb-96c3-2b476d945a96,None
28,4b8d6084-f6b0-4d01-a6ba-9e62ae0b43f6|2b1d454b-...,4b8d6084-f6b0-4d01-a6ba-9e62ae0b43f6,healthcare2_msft_silver,,Lakehouse,2b1d454b-61a1-4abb-96c3-2b476d945a96,None
29,73527721-9956-4aae-8c55-4663fa587fdd|2b1d454b-...,73527721-9956-4aae-8c55-4663fa587fdd,healthcare2_msft_bronze,,Lakehouse,2b1d454b-61a1-4abb-96c3-2b476d945a96,None
37,db5d8fb6-69a0-46d9-a548-64bca94aaa6a|2b1d454b-...,db5d8fb6-69a0-46d9-a548-64bca94aaa6a,InsightWorkbenchAnalytics,,Lakehouse,2b1d454b-61a1-4abb-96c3-2b476d945a96,None
45,daf9c9c0-b2c9-4c8c-aac8-15e0dd058fc4|2b1d454b-...,daf9c9c0-b2c9-4c8c-aac8-15e0dd058fc4,StagingLakehouseForDataflows_20260530091505,,Lakehouse,2b1d454b-61a1-4abb-96c3-2b476d945a96,None


In [21]:
df_lakehouses = pd.DataFrame()

for ws in WORKSPACE_IDs:
    df_new = pd.DataFrame()
    df_new = labs.list_lakehouses(workspace=ws)
    df_new['workspace_id'] = ws

    if (df_lakehouses.empty):
        df_lakehouses = df_new
    else:
        df_lakehouses=pd.concat([df_lakehouses, df_new], axis=0, ignore_index=True)

df_lakehouses = df_lakehouses.astype(str)
df_lakehouse = sanitize_column_names(df_lakehouses)

spark.createDataFrame(df_lakehouses).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("t_lakehouses")

print(f"Created t_lakehouses with {len(df_lakehouses)} lakehouses")
display(df_lakehouses)


StatementMeta(, 3291c773-d509-49e5-a363-18b55b344d05, 28, Finished, Available, Finished, False)

Created t_lakehouses with 6 lakehouses


SynapseWidget(Synapse.DataFrame, c77ee68b-8ad3-4f03-9286-733c45772ec4)

## Get Lakehouse Tables

In [22]:
df_lakehouse_tables = pd.DataFrame()

for index, row in df_lakehouses.iterrows():
    try:
        print(f"Processing lakehouse {row['lakehouse_id']} in workspace {row['workspace_id']}")
        df_new = pd.DataFrame()
        df_new = labs.lakehouse.get_lakehouse_tables(lakehouse=row['lakehouse_id'], workspace=row['workspace_id'])
        df_new['lakehouse_id'] = row['lakehouse_id']

        df_new['workspace_id'] = row['workspace_id']
        
        if (df_lakehouse_tables.empty):
            df_lakehouse_tables = df_new
        else:
            df_lakehouse_tables = pd.concat([df_lakehouse_tables, df_new], axis=0, ignore_index=True)
    except Exception as e:
        print(f"Error processing lakehouse {row['id']} in workspace {row['workspace_id']}: {e}")

df_lakehouse_tables = sanitize_column_names(df_lakehouse_tables)
df_lakehouse_tables = add_composite_key(df_lakehouse_tables, ['table_name', 'lakehouse_id'], 'lakehouse_table_pk')

# Convert all columns to strings to avoid Spark type inference issues
df_lakehouse_tables = df_lakehouse_tables.astype(str)
df_lakehouse_tables = df_lakehouse_tables.replace('nan', None)

spark.createDataFrame(df_lakehouse_tables).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("t_lakehouse_tables")

print(f"Created t_lakehouse_tables with {len(df_lakehouse_tables)} tables")
display(df_lakehouse_tables)

StatementMeta(, 3291c773-d509-49e5-a363-18b55b344d05, 29, Finished, Available, Finished, False)

Processing lakehouse b357c6a6-b3dd-46bd-beb7-32bbe8f9b637 in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing lakehouse d555b438-935c-4a50-a14b-93d637712a30 in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing lakehouse 4b8d6084-f6b0-4d01-a6ba-9e62ae0b43f6 in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing lakehouse 73527721-9956-4aae-8c55-4663fa587fdd in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing lakehouse db5d8fb6-69a0-46d9-a548-64bca94aaa6a in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing lakehouse daf9c9c0-b2c9-4c8c-aac8-15e0dd058fc4 in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Created t_lakehouse_tables with 200 tables


SynapseWidget(Synapse.DataFrame, 7f50464c-1bfd-4259-9f5b-b651dc4c23e8)

## Get Lakehouse Columns

In [ ]:
df_lakehouse_columns = pd.DataFrame()

for index, row in df_lakehouses.iterrows():
    try:
        print(f"Processing lakehouse {row['lakehouse_id']} in workspace {row['workspace_id']}")
        df_new = pd.DataFrame()
        df_new = labs.lakehouse.get_lakehouse_columns(lakehouse=row['lakehouse_id'], workspace=row['workspace_id'])
        df_new['lakehouse_id'] = row['lakehouse_id']
        df_new['workspace_id'] = row['workspace_id']
        
        if (df_lakehouse_columns.empty):
            df_lakehouse_columns = df_new
        else:
            df_lakehouse_columns = pd.concat([df_lakehouse_columns, df_new], axis=0, ignore_index=True)
    except Exception as e:
        print(f"Error processing lakehouse {row['id']} in workspace {row['workspace_id']}: {e}")

df_lakehouse_columns = sanitize_column_names(df_lakehouse_columns)
df_lakehouse_columns = add_composite_key(df_lakehouse_columns, ['table_name', 'column_name', 'lakehouse_id'], 'lakehouse_column_pk')

# Convert all columns to strings to avoid Spark type inference issues
df_lakehouse_columns = df_lakehouse_columns.astype(str)
df_lakehouse_columns = df_lakehouse_columns.replace('nan', None)

spark.createDataFrame(df_lakehouse_columns).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("t_lakehouse_columns")

print(f"Created t_lakehouse_columns with {len(df_lakehouse_columns)} columns")
display(df_lakehouse_columns)

StatementMeta(, 6dfa0e71-6c23-4894-91ee-d02250b872db, 73, Finished, Available, Finished, False)

Processing lakehouse b357c6a6-b3dd-46bd-beb7-32bbe8f9b637 in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing lakehouse d555b438-935c-4a50-a14b-93d637712a30 in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing lakehouse 4b8d6084-f6b0-4d01-a6ba-9e62ae0b43f6 in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing lakehouse 73527721-9956-4aae-8c55-4663fa587fdd in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing lakehouse db5d8fb6-69a0-46d9-a548-64bca94aaa6a in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing lakehouse daf9c9c0-b2c9-4c8c-aac8-15e0dd058fc4 in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Created t_lakehouse_columns with 8148 columns


SynapseWidget(Synapse.DataFrame, b55086f0-9684-4d4e-b873-0e731af2f17e)

StatementMeta(, 3291c773-d509-49e5-a363-18b55b344d05, 30, Finished, Available, Finished, False)

Processing lakehouse b357c6a6-b3dd-46bd-beb7-32bbe8f9b637 in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing lakehouse d555b438-935c-4a50-a14b-93d637712a30 in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing lakehouse 4b8d6084-f6b0-4d01-a6ba-9e62ae0b43f6 in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing lakehouse 73527721-9956-4aae-8c55-4663fa587fdd in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing lakehouse db5d8fb6-69a0-46d9-a548-64bca94aaa6a in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Processing lakehouse daf9c9c0-b2c9-4c8c-aac8-15e0dd058fc4 in workspace 2b1d454b-61a1-4abb-96c3-2b476d945a96
Created t_lakehouse_columns with 8148 columns


SynapseWidget(Synapse.DataFrame, 329a4601-952a-45c0-909d-002e946b513c)